# Assignment 1: N-Gram Language Models

In lecture, we defined language modeling as the task of predicting the next word in a sequence given the previous words. In this assignment, we will focus on the related problem of predicting the next character in a sequence given the previous characters.

The learning goals of this assignment are to:

- Understand how to compute language model probabilities using maximum likelihood estimation.
- Implement basic smoothing, back-off and interpolation.
- Have fun using a language model to probabilistically generate texts!
- Use a set of language models to perform text classification.

## Submission Details

Make sure to submit this python notebook along with any necessary text files needed to run your code to Brightspace by **10/05/2026 11:59pm EST**.

In [ ]:
import math, random
from collections import defaultdict

In [ ]:
################################################################################
# Utility Functions
################################################################################

COUNTRY_CODES = ['af', 'cn', 'de', 'fi', 'fr', 'in', 'ir', 'pk', 'za']

def start_pad(n):
    ''' Returns a padding string of length n to append to the front of text
        as a pre-processing step to building n-grams '''
    return '~' * n

def create_ngram_model(model_class, path, n=2, k=0):
    ''' Creates and returns a new n-gram model trained on the city names
        found in the path file '''
    model = model_class(n, k)
    with open(path, encoding='utf-8', errors='ignore') as f:
        model.update(f.read())
    return model

def create_ngram_model_lines(model_class, path, n=2, k=0):
    ''' Creates and returns a new n-gram model trained on the city names
        found in the path file '''
    model = model_class(n, k)
    with open(path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            model.update(line.strip())
    return model

## Part 1: Generating N-Grams (4 pts)


Write a function `ngrams(n, text)` that produces a list of all n-grams of size `n` from the given `text`.  Each n-gram should consist of a 2-element tuple `(context, char)` where `context` is the `n`-length string comprised of the `n` charachters preceding `char`.

Make sure to pad the beginning of the sentence with `n` ~ charachters (I have included a function `start_pad` above for you to use).  If `n=0`, you will always have `context = ""`.  Assume that `n>= 0`.

Make sure to test your function with some examples.

e.g.
```python
>>> ngrams(1, 'abc')
[('~', 'a'), ('a', 'b'), ('b', 'c')]

>>> ngrams(2, 'abc')
[('~~', 'a'), ('~a', 'b'), ('ab', 'c')]
```

In [ ]:
def ngrams(n, text):
    ''' Returns the ngrams of the text as tuples where the first element is
        the length-n context and the second is the character '''
    ### TODO ###
    padded_text = start_pad(n) + text
    result = []

    for i in range(len(text)):
        context = padded_text[i : i+n]
        char = padded_text[i+n]
        result.append((context, char))

    return result
    pass


## Part 2: Creating an N-gram Model (20 pts)

You will build a simple n-gram language model that can be used to generate some random text that resembles a source document. Your use of external code should be limited to built-in Python modules, which excludes, for example, NumPy and NLTK.

In the `NgramModel` class, I have included an initialization method `__init__(self, n, k)` which stores the order `n` of the model and initializes any necessary internal variables.

1. Write a method `get_vocab(self)` that returns the vocab (this is the set of all characters used by this model).

2. Write a method `update(self, text)` which computes the n-grams for the input sentence and updates the internal counts.
3. Also write a method `prob(self, context, char)` which accepts an `n`-length string representing a context and a character, and returns the probability of that character occuring, given the preceding context. If you encounter a novel `context`, the probability of any given `char` should be $1/V$ where $V$ is the size of the vocab.

Make sure to test the functionality of your code.

```python
 >>> m = NgramModel(1, 0)
 >>> m.update('abab')
 >>> m.get_vocab()
 {'b', 'a'}
 >>> m.update('abcd')
 >>> m.get_vocab()
 {'b', 'a', 'c', 'd'}
 >>> m.prob('a', 'b')
 1.0
 >>> m.prob('~', 'c')
 0.0
 >>> m.prob('b', 'c')
 0.5
```


In [ ]:
################################################################################
# Basic N-Gram Model
################################################################################

class NgramModel(object):
    ''' A basic n-gram model using add-k smoothing '''

    def __init__(self, n, k):
        self.n = n
        self.k = k
        self.vocab = set()
        self.context_counts = defaultdict(lambda:0)
        self.sequence_counts = defaultdict(lambda:0)

    def get_vocab(self):
        ''' Returns the set of characters in the vocab '''
        ### TODO ###
        return self.vocab
        pass

    def update(self, text):
        ''' Updates the model n-grams based on text '''
        ### TODO ###
        self.vocab.update(list(text))
        for context, char in ngrams(self.n, text):
            self.context_counts[context] += 1
            self.sequence_counts[context + char] += 1
        pass

    def prob(self, context, char):
        ''' Returns the probability of char appearing after context (without smoothing) '''
        ### TODO ###
        vocab_size = len(self.vocab)
        if vocab_size == 0:
            return 0.0

        count_context = self.context_counts[context]
        count_sequence = self.sequence_counts[context + char]

        if count_context == 0:
            return 1.0 / vocab_size

        return count_sequence / count_context
        pass

In [ ]:
### Test your code here ###

I have provided a method `random_char(self, context)` which returns a random character according to the probability distribution determined by the given context. Specifically, let $V = \{w_1, w_2, \ldots, w_n\}$ be the vocab, sorted according to Python's natural lexicographic ordering, and let $0\leq r \leq 1$ be a random number between $0$ and $1$. The method returns a character $w_i$ such that
$$\sum_{j=1}^{i-1}P(w_j|\text{context}) \leq r < \sum_{j=1}^iP(w_j|\text{context})$$

In [ ]:
def random_char(self, context):
    ''' Returns a random character based on the given context and the
        n-grams learned by this model '''
    r = random.random()
    pre_sum = 0
    for i, char in enumerate(sorted(self.vocab)):
        post_sum = pre_sum + self.prob(context, char)
        if pre_sum <= r < post_sum:
            return char
        pre_sum = post_sum

NgramModel.random_char = random_char

In [ ]:
### Feel free to uncomment the following code to see how the method works ###
# m = NgramModel(0, 0)
# m.update('abab')
# m.update('abcd')
# random.seed(1)
# [m.random_char('') for i in range(25)]

4. Implement the method `random_text(self, length)` which returns a string of characters chosen at random using the supplied `random_char(self, context)` method.  Your starting context should always be `n` ~ characters, and the context should be updated as characters are generated. If `n=0`, your context should always be the empty string. You should continue generating characters until you've produced the specified number of random characters, then return the full string.

Once again, be sure to test your code.  You should be able to achieve something like the following

```python
 >>> m = NgramModel(1, 0)
 >>> m.update('abab')
 >>> m.update('abcd')
 >>> random.seed(1)
 >>> m.random_text(25)
 abcdbabcdabababcdddabcdba
```

In [ ]:
def random_text(self, length):
    ''' Returns text of the specified character length based on the
        n-grams learned by this model '''
    ### TODO ###
    context = start_pad(self.n)
    generated_text = ""

    for _ in range(length):
        char = self.random_char(context)
        generated_text += char

        if self.n > 0:
            context = (context + char)[-self.n:]

    return generated_text

NgramModel.random_text = random_text


NgramModel.random_text = random_text

### Writing Shakespeare

Now you're ready to train your first language model!  First let's grab some text:

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-05 22:50:17--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-10-05 22:50:17 (20.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



Try generating some Shakespeare with different order n-gram models. You should try running the following commands:

In [ ]:
m = create_ngram_model(NgramModel, 'input.txt', 2)
m.random_text(250)

'Fir, Devieday, Claings.\n\nQUE:\nOwell, fole,\nABETH:\nGRENE:\nNot he hen thim will def of award a fainstat ightlew-dameturstand do ame too.\n\nKINGBRUTOLIZABENIUS:\nWARD:\n\nBUCEN EDWAREN:\nProve re petheard:\nWhose mostakinevir en percy me tor lou ar. Telost at'

In [ ]:
m = create_ngram_model(NgramModel, 'input.txt', 3)
m.random_text(250)

"First shall.\n\nWARD ROSPERO:\nSay cour goss soung slave fough 'gainto heavy king so you.\n\nPOLIO:\nIt is my sworse\nBy a stitle a world, a more.\n\nTHOMAS:\nWell haca pluck\nHave Henry fallo will kingalls woes of mildress Brous hear tillainstraintently. Provo"

In [ ]:
m = create_ngram_model(NgramModel, 'input.txt', 4)
m.random_text(250)

'First: for I wound, from him\nIn the Lady your words, not guilties your women cousin now he the on her bashful succording mammocking thee, lady Bona, the blind, to his times, I this clearn of my conclude a tyrant;\nFor thin the two,\nTo make crave.\n\nFir'

In [ ]:
m = create_ngram_model(NgramModel, 'input.txt', 7)
m.random_text(250)

"First Citizen:\nHe shall we desire his leave\nTo look upon my heart to that e'er till now:\nNow shall satisfaction on himself:\nSo cunning in the throne into this after thee:\nThat no discipline,\nLet villain in the figured clean:\nYou are deceit bred this "

After generating a bunch of short passages, do you notice anything? Write down some of your observations about the language model.

#### Observations

**TODO**: write down observations about the model

## Part 3: Perplexity, Smoothing, and Interpolation (6 pts)

### Perplexity
How do we know whether our language model is good?  As we talked about in lecture, we use an intrinsic evaluation method called **perplexity**.

In this part of the assignment, you will implement the `perplexity(self, text)` method.  A few things to keep in mind:
- Numeric underflow will most likely be a problem, so consider using logs (as discussed in lecture)
- Perplexity is undefined if the language model assigns any zero probabilities to the test set. In that case your code should return positive infinity - `float('inf')`.
- On your unsmoothed models, you'll definitely get some zero probabilities for the test set. To test you code, you should try computing perplexity on the training set, and you should compute perplexity for your language models that use smoothing and interpolation.

Be sure to test your code

```python
>>> m = NgramModel(1, 0)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.perplexity('abcd')
1.189207115002721
>>> m.perplexity('abca')
inf
>>> m.perplexity('abcda')
1.515716566510398
```

In [ ]:
def perplexity(self, text):
    ''' Returns the perplexity of text based on the n-grams learned by
        this model '''
    ### TODO ###
    log_prob_sum = 0.0
    N = len(text)

    if N == 0:
        return 0.0

    for context, char in ngrams(self.n, text):
        p = self.prob(context, char)
        if p == 0:
            return float('inf')
        log_prob_sum += math.log(p)

    return math.exp(-log_prob_sum / N)
    pass

NgramModel.perplexity = perplexity

Discuss the perplexity for text that is similar and different from Shakespeare's plays.  

You are provided two dev text files, a New York Times article and several of Shakespeare's sonnets.  Though you are free to experiment with any text of your choosing.

In [ ]:
# download and unzip the data
!wget https://computational-linguistics-class.org/homework/ngram-lms/test_data.zip
!unzip test_data.zip

--2026-10-05 22:50:47--  https://computational-linguistics-class.org/homework/ngram-lms/test_data.zip
Resolving computational-linguistics-class.org (computational-linguistics-class.org)... 185.199.110.153
Connecting to computational-linguistics-class.org (computational-linguistics-class.org)|185.199.110.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6998 (6.8K) [application/x-zip-compressed]
Saving to: ‘test_data.zip’

test_data.zip       100%[===================>]   6.83K  --.-KB/s    in 0s      

2026-10-05 22:50:48 (52.6 MB/s) - ‘test_data.zip’ saved [6998/6998]

Archive:  test_data.zip
  inflating: nytimes_article.txt     
  inflating: shakespeare_sonnets.txt  


In [ ]:
### TODO: investigate the perplexity for the two different texts ###
# Read the text from the downloaded files with errors ignored to avoid decoding issues
with open('nytimes_article.txt', 'r', encoding='utf-8', errors='ignore') as f:
    nyt_text = f.read()

with open('shakespeare_sonnets.txt', 'r', encoding='utf-8', errors='ignore') as f:
    sonnets_text = f.read()

# Calculate perplexity using your most recently trained model 'm'
print(f"Shakespeare Sonnets Perplexity: {m.perplexity(sonnets_text)}")
print(f"New York Times Perplexity: {m.perplexity(nyt_text)}")

Shakespeare Sonnets Perplexity: inf
New York Times Perplexity: inf


### Smoothing
Laplace Smoothing is described in section 3.5.1 of the [textbook](https://web.stanford.edu/~jurafsky/slp3/).  It adds 1 to each count, and since there are $V$ characters in the vocabulary and each one was incremented, we adjust the denominator to take this into account as well.

$$P_{\text{Laplace}}(w_i) = \frac{Count(w_i) + 1}{N + |V|}$$

A variant of Laplace smoothing is called Add-k smoothing or Add-epsilon smoothing. This is described in section 3.5.2 of the textbook. Update your implementation of the `prob(self, context, char)` method from Part 1 to include add-k smoothing.

Be sure to test your code

```python
>>> m = NgramModel(1, 1)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.prob('a', 'a')
0.14285714285714285
>>> m.prob('a', 'b')
0.5714285714285714
>>> m.prob('c', 'd')
0.4
>>> m.prob('d', 'a')
0.25
```


In [ ]:
def prob(self, context, char):
    ''' Returns the probability of char appearing after context with add-k smoothing '''
    ### TODO ###
    vocab_size = len(self.vocab)
    if vocab_size == 0:
        return 0.0

    count_context = self.context_counts[context]
    count_sequence = self.sequence_counts[context + char]

    if self.k == 0:
        if count_context == 0:
            return 1.0 / vocab_size
        return count_sequence / count_context
    else:
        return (count_sequence + self.k) / (count_context + self.k * vocab_size)
    pass

NgramModel.prob = prob

### Interpolation (Optional)

The idea of interpolation is to calculate the higher order n-gram probabilities also combining the probabilities for lower-order n-gram models. Like smoothing, this helps us avoid the problem of zeros if we haven't observed the longer sequence in our training data.

$$P_{\text{interpolation}}(w_i|w_{i-2}w_{i-1}) = \lambda_1P(w_i|w_{i-2}w_{i-1}) + \lambda_2P(w_i|w_{i-1}) + \lambda_3P(w_i)$$

Below is another class definition, `NGramModelWithInterpolation` that extends the previously defined class `NGramModel`.  If you've written your code robustly, you should only need to override the following functions
- `get_vocab(self)`
- `update(self, text)`
- `prob(self, context, char)`

The value of `n` passed into the constructor is the highest order n-gram to be considered by the model (e.g. `n=2` will consider 3 different length n-grams: 2, 1, and 0).  Add-k smoothing should only occur when calculating the individual n-gram probabilities, not when calculating the overall interpolation probability!

By default set the lambdas to be equal weights, but you should also write a helper function that can be called to overwrite this default. Setting the lambdas in the helper function can either be done heuristically or by using a development set, but in the example code below, we've used the default.

```python
>>> m = NgramModelWithInterpolation(1, 0)
>>> m.update('abab')
>>> m.prob('a', 'a')
0.25
>>> m.prob('a', 'b')
0.75

>>> m = NgramModelWithInterpolation(2, 1)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.prob('~a', 'b')
0.4682539682539682
>>> m.prob('ba', 'b')
0.4349206349206349
>>> m.prob('~c', 'd')
0.27222222222222225
>>> m.prob('bc', 'd')
0.3222222222222222
```

In [ ]:
################################################################################
# N-Gram Model with Interpolation
################################################################################

class NgramModelWithInterpolation(NgramModel):
    ''' An n-gram model with interpolation '''

    def __init__(self, n, k):
        pass

    def get_vocab(self):
        pass

    def update(self, text):
        pass

    def prob(self, context, char):
        pass

Now that you've created an N-gram model with interpolation, experiment with a few different lambdas and values of `k` and discuss their effects below.